# 03. Lag Analysis

전처리 통합 데이터와 02 단계에서 선택한 국제제품 benchmark를 사용해 국내 가격 반영 시차를 추정합니다.

이 노트북은 Google Colab에서 단독 실행되도록 구성되어 있습니다. 첫 설정 셀의 경로만 본인 Google Drive 구조에 맞게 수정한 뒤 위에서 아래로 실행하면 됩니다.


In [ ]:
# ============================================================
# 03 공통 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = os.path.join(ROOT_PATH, "data") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"

print(f"ROOT_PATH      = {ROOT_PATH}")
print(f"DATA_PATH      = {DATA_PATH}")
print(f"PROCESSED_PATH = {PROCESSED_PATH}")
LAG_OUTPUT_PATH = os.path.join(ROOT_PATH, "시차분석_v2") + "/"
print(f"LAG_OUTPUT_PATH = {LAG_OUTPUT_PATH}")

# 필요한 패키지 확인/설치
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "statsmodels": "statsmodels",
    "tqdm": "tqdm",
    "IPython": "ipython",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"[설치] 필요한 패키지 설치: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("[확인] 필요한 패키지가 이미 설치되어 있습니다.")


In [ ]:
# =========================================================
# 1단계 공통 설정
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# 결과 저장 폴더
result_file_name = "시차분석_v2"

# 분석 기간
ANALYSIS_START = "2008-05-10"
ANALYSIS_END   = "2025-12-31"

# 정유소 sparse 주간 시계열의 첫 실제 week_end
FIRST_REFINERY_WEEK_END = "2008-05-10"

# 유종별 기본 benchmark
# 각 분석 셀에서 benchmark_col을 직접 넘기면 이 기본값은 무시된다.
DEFAULT_BENCHMARK_BY_FUEL = {
    "gasoline": "휘발유92RON_원리터",
    "diesel": "경유0.001_원리터",
}

# 유종별 컬럼 설정
STAGE1_FUEL_CONFIG = {
    "gasoline": {
        "label": "휘발유",
        "refinery_pretax_col": "정유소_세전_보통휘발유",
        "retail_gross_col": "보통휘발유_평균",
        "tax_cols": [
            "보통휘발유_개별소비세",
            "보통휘발유_교통에너지환경세",
            "보통휘발유_교육세",
            "보통휘발유_주행세",
            "보통휘발유_판매부과금",
        ],
        "brand_cols": [
            "보통휘발유_정유사평균",
            "보통휘발유_SK에너지",
            "보통휘발유_GS칼텍스",
            "보통휘발유_HD현대오일뱅크",
            "보통휘발유_S-OIL",
            "보통휘발유_알뜰주유소",
            "보통휘발유_알뜰-자영",
            "보통휘발유_자가상표",
        ],
    },
    "diesel": {
        "label": "경유",
        "refinery_pretax_col": "정유소_세전_자동차용경유",
        "retail_gross_col": "자동차용경유_평균",
        "tax_cols": [
            "자동차용경유_개별소비세",
            "자동차용경유_교통에너지환경세",
            "자동차용경유_교육세",
            "자동차용경유_주행세",
            "자동차용경유_판매부과금",
        ],
        "brand_cols": [
            "자동차용경유_정유사평균",
            "자동차용경유_SK에너지",
            "자동차용경유_GS칼텍스",
            "자동차용경유_HD현대오일뱅크",
            "자동차용경유_S-OIL",
            "자동차용경유_알뜰주유소",
            "자동차용경유_알뜰-자영",
            "자동차용경유_자가상표",
        ],
    },
}

# 정책/충격 제외기간
EXCLUDE_RANGES_STAGE1 = [
    {"start": "2008-03-10", "end": "2008-12-31", "label": "fuel_tax_cut_2008"},
    {"start": "2011-04-07", "end": "2011-07-06", "label": "유류세 100원 인하"},
    {"start": "2018-11-06", "end": "2019-05-06", "label": "fuel_tax_cut_2018_15pct"},
    {"start": "2019-05-07", "end": "2019-08-31", "label": "fuel_tax_cut_2019_7pct"},
    {"start": "2021-11-12", "end": "2022-04-30", "label": "fuel_tax_cut_2021_20pct"},
    {"start": "2022-05-01", "end": "2022-06-30", "label": "fuel_tax_cut_2022_30pct"},
    {"start": "2022-07-01", "end": "2022-12-31", "label": "fuel_tax_cut_2022_37pct"},
    {"start": "2023-01-01", "end": "2024-06-30", "label": "fuel_tax_cut_maintained_2023_2024"},
    {"start": "2024-07-01", "end": "2024-10-31", "label": "fuel_tax_cut_2024_partial"},
    {"start": "2024-11-01", "end": "2025-04-30", "label": "fuel_tax_cut_2024_2025_partial"},
    {"start": "2025-05-01", "end": "2025-10-31", "label": "fuel_tax_cut_2025_readjusted"},
    {"start": "2025-11-01", "end": "2026-04-30", "label": "fuel_tax_cut_2025_2026_partial"},
    {"start": "2026-03-13", "end": "2026-03-26", "label": "price_cap_2026_round1"},
]

print("[READY] 1단계 공통 설정 완료")
print(f"ANALYSIS_START = {ANALYSIS_START}")
print(f"ANALYSIS_END   = {ANALYSIS_END}")
print("유종은 각 분석 셀에서 FUEL_KEY로 선택합니다.")

In [ ]:
# =========================================================
# 1단계 frame 생성 함수
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


def resolve_integrated_path(processed_path) -> Path:
    base = Path(processed_path)
    candidates = [
        base / "분석용_일별_통합데이터.csv",
        base / "분석용_일별_통합데이터.csv",
    ]

    for c in candidates:
        if c.exists():
            return c

    matches = sorted(base.glob("*통합데이터*.csv"))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"분석용 일별 통합데이터 CSV를 찾지 못했습니다. PROCESSED_PATH={processed_path}"
    )


INTEGRATED_PATH = resolve_integrated_path(PROCESSED_PATH)

lag_df = pd.read_csv(INTEGRATED_PATH)
lag_df["date"] = pd.to_datetime(lag_df["date"], errors="coerce")
lag_df = lag_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

# 기존 코드 호환용
df = lag_df.copy()


def rebuild_refinery_weekly_from_order(
    df: pd.DataFrame,
    value_col: str,
    first_week_end: str = FIRST_REFINERY_WEEK_END,
) -> pd.DataFrame:
    tmp = (
        df.loc[df[value_col].notna(), ["date", value_col]]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    if len(tmp) == 0:
        raise ValueError(f"[정유소 주간값 없음] {value_col}")

    week_index = pd.date_range(
        start=pd.Timestamp(first_week_end),
        periods=len(tmp),
        freq="W-SAT",
    )

    return pd.DataFrame({
        "date": week_index,
        value_col: pd.to_numeric(tmp[value_col], errors="coerce").values,
        "source_date_in_csv": tmp["date"].values,
    })


def weekly_mean_from_daily(
    df: pd.DataFrame,
    value_col: str,
    rule: str = "W-SAT",
) -> pd.DataFrame:
    s = (
        df[["date", value_col]]
        .dropna(subset=[value_col])
        .copy()
        .set_index("date")[value_col]
        .astype(float)
        .sort_index()
    )

    return s.resample(rule).mean().rename(value_col).reset_index()


def build_retail_pretax_equiv_daily(
    df: pd.DataFrame,
    retail_gross_col: str,
    tax_cols: list,
    out_col: str,
) -> pd.DataFrame:
    work_cols = ["date", retail_gross_col] + tax_cols
    work = df[work_cols].copy()

    if tax_cols:
        work[tax_cols] = work[tax_cols].ffill()
        tax_sum = work[tax_cols].fillna(0).sum(axis=1)
    else:
        tax_sum = 0.0

    # 기존 1단계 보조 frame 호환용.
    # 현재 main 소비자 시차는 gross price를 사용한다.
    work[out_col] = work[retail_gross_col] / 1.1 - tax_sum

    return work[["date", out_col]]


def trim_by_period(df_: pd.DataFrame) -> pd.DataFrame:
    out = df_.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")

    start_ts = pd.Timestamp(ANALYSIS_START)
    end_ts = pd.Timestamp(ANALYSIS_END)

    return out[
        (out["date"] >= start_ts) &
        (out["date"] <= end_ts)
    ].reset_index(drop=True)


def resolve_stage1_fuel_setting(
    fuel_key: str,
    benchmark_col: str | None = None,
) -> dict:
    if fuel_key not in STAGE1_FUEL_CONFIG:
        raise ValueError(f"fuel_key는 {list(STAGE1_FUEL_CONFIG.keys())} 중 하나여야 합니다. 입력값={fuel_key}")

    cfg = STAGE1_FUEL_CONFIG[fuel_key].copy()
    cfg["fuel_key"] = fuel_key
    cfg["benchmark_col"] = benchmark_col or DEFAULT_BENCHMARK_BY_FUEL[fuel_key]

    return cfg


def prepare_stage1_frames(
    fuel_key: str,
    benchmark_col: str | None = None,
    *,
    verbose: bool = True,
) -> dict:
    """
    특정 유종/benchmark에 대한 1단계 분석용 frame 생성.

    반환 frame:
      - consumer_daily_gross_df : 국제제품가격 daily -> 주유소 소비자가격 daily
      - consumer_daily_pretax_df: 보조 frame
      - weekly_upstream_df      : 국제제품가격 weekly -> 정유사 세전공급가격 weekly
      - weekly_downstream_df    : 정유사 세전공급가격 weekly -> 주유소가격 weekly 보조 frame
      - weekly_direct_df        : 국제제품가격 weekly -> 주유소가격 weekly 보조 frame
      - daily_operating_df      : 향후 2단계 연결용 보조 frame
      - brand_weekly_df         : 브랜드별 주간 평균 보조 frame
    """
    cfg = resolve_stage1_fuel_setting(fuel_key, benchmark_col)

    refinery_pretax_col = cfg["refinery_pretax_col"]
    retail_gross_col = cfg["retail_gross_col"]
    benchmark_col = cfg["benchmark_col"]

    tax_cols = [c for c in cfg["tax_cols"] if c in lag_df.columns]
    brand_cols = [c for c in cfg["brand_cols"] if c in lag_df.columns]

    required_cols = ["date", benchmark_col, refinery_pretax_col, retail_gross_col]
    missing_cols = [c for c in required_cols if c not in lag_df.columns]
    if missing_cols:
        raise ValueError(f"[시차분석 필수 컬럼 누락] {missing_cols}")

    retail_pretax_daily_col = retail_gross_col.replace("_평균", "_세전환산")

    retail_pretax_daily_df = build_retail_pretax_equiv_daily(
        lag_df,
        retail_gross_col=retail_gross_col,
        tax_cols=tax_cols,
        out_col=retail_pretax_daily_col,
    )

    # -----------------------------------------------------
    # 1-A: 소비자 체감 시차용 daily frame
    # -----------------------------------------------------
    consumer_daily_gross_df = (
        lag_df[["date", benchmark_col, retail_gross_col]]
        .rename(columns={
            benchmark_col: "benchmark_daily",
            retail_gross_col: "retail_gross_daily",
        })
        .sort_values("date")
        .reset_index(drop=True)
    )

    consumer_daily_pretax_df = (
        lag_df[["date", benchmark_col]]
        .rename(columns={benchmark_col: "benchmark_daily"})
        .merge(
            retail_pretax_daily_df.rename(columns={retail_pretax_daily_col: "retail_pretax_daily"}),
            on="date",
            how="left",
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 1-B: 정유사 upstream weekly frame
    # -----------------------------------------------------
    benchmark_weekly_df = weekly_mean_from_daily(
        lag_df,
        benchmark_col,
        rule="W-SAT",
    ).rename(columns={benchmark_col: "benchmark_weekly"})

    refinery_weekly_df = rebuild_refinery_weekly_from_order(
        lag_df,
        value_col=refinery_pretax_col,
        first_week_end=FIRST_REFINERY_WEEK_END,
    ).rename(columns={refinery_pretax_col: "refinery_pre_tax_weekly"})

    weekly_upstream_df = (
        benchmark_weekly_df
        .merge(
            refinery_weekly_df[["date", "refinery_pre_tax_weekly"]],
            on="date",
            how="inner",
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 1-C: downstream 보조 weekly frame
    # -----------------------------------------------------
    retail_pretax_weekly_df = weekly_mean_from_daily(
        retail_pretax_daily_df,
        retail_pretax_daily_col,
        rule="W-SAT",
    ).rename(columns={retail_pretax_daily_col: "retail_pretax_equiv_weekly"})

    retail_gross_weekly_df = weekly_mean_from_daily(
        lag_df,
        retail_gross_col,
        rule="W-SAT",
    ).rename(columns={retail_gross_col: "retail_gross_weekly"})

    weekly_downstream_df = (
        refinery_weekly_df[["date", "refinery_pre_tax_weekly"]]
        .merge(retail_pretax_weekly_df, on="date", how="inner")
        .merge(retail_gross_weekly_df, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 1-D: direct weekly reduced-form check
    # -----------------------------------------------------
    weekly_direct_df = (
        benchmark_weekly_df
        .merge(retail_pretax_weekly_df, on="date", how="inner")
        .merge(retail_gross_weekly_df, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 향후 2단계 연결용 operating frame
    # -----------------------------------------------------
    daily_operating_cols = ["date", benchmark_col, retail_gross_col] + tax_cols

    daily_operating_df = (
        lag_df[daily_operating_cols]
        .rename(columns={
            benchmark_col: "benchmark_daily",
            retail_gross_col: "retail_gross_daily",
        })
        .merge(
            retail_pretax_daily_df.rename(columns={retail_pretax_daily_col: "retail_pretax_daily"}),
            on="date",
            how="left",
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    if brand_cols:
        brand_weekly_df = (
            lag_df.set_index("date")[brand_cols]
            .resample("W-SAT")
            .mean()
            .reset_index()
        )
    else:
        brand_weekly_df = pd.DataFrame(columns=["date"])

    # 기간 자르기
    consumer_daily_gross_df = trim_by_period(consumer_daily_gross_df)
    consumer_daily_pretax_df = trim_by_period(consumer_daily_pretax_df)
    weekly_upstream_df = trim_by_period(weekly_upstream_df)
    weekly_downstream_df = trim_by_period(weekly_downstream_df)
    weekly_direct_df = trim_by_period(weekly_direct_df)
    daily_operating_df = trim_by_period(daily_operating_df)

    if len(brand_weekly_df) > 0:
        brand_weekly_df = trim_by_period(brand_weekly_df)

    frames = {
        "fuel_key": fuel_key,
        "fuel_label": cfg["label"],
        "benchmark_col": benchmark_col,
        "refinery_pretax_col": refinery_pretax_col,
        "retail_gross_col": retail_gross_col,
        "tax_cols": tax_cols,
        "brand_cols": brand_cols,
        "consumer_daily_gross_df": consumer_daily_gross_df,
        "consumer_daily_pretax_df": consumer_daily_pretax_df,
        "weekly_upstream_df": weekly_upstream_df,
        "weekly_downstream_df": weekly_downstream_df,
        "weekly_direct_df": weekly_direct_df,
        "daily_operating_df": daily_operating_df,
        "brand_weekly_df": brand_weekly_df,
    }

    if verbose:
        print("=" * 100)
        print("[1단계 frame 생성 완료]")
        print(f"fuel_key              : {fuel_key}")
        print(f"fuel_label            : {cfg['label']}")
        print(f"benchmark_col         : {benchmark_col}")
        print(f"refinery_pretax_col   : {refinery_pretax_col}")
        print(f"retail_gross_col      : {retail_gross_col}")
        print(f"analysis period       : {ANALYSIS_START} ~ {ANALYSIS_END}")
        print(f"consumer daily rows   : {len(consumer_daily_gross_df):,}")
        print(f"refinery weekly rows  : {len(weekly_upstream_df):,}")
        print(f"tax cols used         : {tax_cols}")
        print("=" * 100)

        print("\n[consumer_daily_gross_df]")
        display(consumer_daily_gross_df.head())

        print("\n[weekly_upstream_df]")
        display(weekly_upstream_df.head())

    return frames


print(f"[INTEGRATED_PATH] {INTEGRATED_PATH}")
print(f"[lag_df] {len(lag_df):,} rows")
print("[READY] prepare_stage1_frames(fuel_key, benchmark_col) 정의 완료")

In [ ]:
# =========================================================
# 0) 유틸
# =========================================================
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
from pathlib import Path
import json



def normalize_exclude_ranges(exclude_ranges: Optional[Sequence] = None) -> List[Tuple[pd.Timestamp, pd.Timestamp]]:
    """
    입력: exclude_ranges=None
    출력: List[Tuple[pd.Timestamp, pd.Timestamp]]
    설명: 제외할 날짜 구간 목록을 시작일-종료일 Timestamp 쌍 목록으로 정규화
    """
    out: List[Tuple[pd.Timestamp, pd.Timestamp]] = []
    for item in (exclude_ranges or []):
        if isinstance(item, dict):
            start = item["start"]
            end = item["end"]
        else:
            start, end = item
        start_ts = pd.Timestamp(start).normalize()
        end_ts = pd.Timestamp(end).normalize()
        if end_ts < start_ts:
            start_ts, end_ts = end_ts, start_ts
        out.append((start_ts, end_ts))
    return out



def _ensure_datetime_index(data: pd.DataFrame) -> pd.DataFrame:
    """
    입력: data: pd.DataFrame
    출력: pd.DataFrame
    설명: date 컬럼 또는 index를 datetime으로 변환하고 날짜 인덱스로 정리
    """
    s = data.copy()
    if "date" in s.columns:
        s["date"] = pd.to_datetime(s["date"], errors="coerce")
        s = s.dropna(subset=["date"]).set_index("date")
    else:
        s.index = pd.to_datetime(s.index, errors="coerce")
        s = s[~s.index.isna()]
    return s.sort_index()



def resample_level_data(
    data: pd.DataFrame,
    dom_col: str,
    intl_col: str,
    rule: str,
    how: str = "mean",
) -> pd.DataFrame:
    """
    입력: data: pd.DataFrame, dom_col: str, intl_col: str, rule: str, how: str="mean"
    출력: pd.DataFrame
    설명: 국내/국제 레벨 데이터를 지정 주기로 평균 또는 마지막 값 기준 집계
    """
    s = _ensure_datetime_index(data)
    s = s[[dom_col, intl_col]].apply(pd.to_numeric, errors="coerce")
    if how == "mean":
        out = s.resample(rule).mean()
    elif how == "last":
        out = s.resample(rule).last()
    else:
        raise ValueError("how 는 'mean' 또는 'last' 여야 합니다.")
    return out

In [ ]:
# =========================================================
# 1) 레벨 -> 변화율 데이터 생성
#    - full calendar index를 유지해 calendar-day lag를 보존
#    - missing fill은 forward-fill만 사용 (bfill 제거)
# =========================================================

def build_change_frame(
    data: pd.DataFrame,
    dom_col: str,
    intl_col: str,
    start: Optional[str] = None,
    end: Optional[str] = None,
    change_method: str = "logdiff",  # "logdiff" 또는 "pct"
    scale: float = 100.0,
    exclude_ranges: Optional[Sequence] = None,
    freq: str = "D",
    fill_method: str = "ffill",
) -> pd.DataFrame:

    """
    data: pd.DataFrame - 날짜와 분석 대상 값들이 들어 있는 원본 데이터프레임
    dom_col: str - 국내 가격 컬럼명 문자열
    intl_col: str - 국제 가격 컬럼명 문자열
    start: Optional[str] - 분석 시작일 문자열 또는 None
    end: Optional[str] - 분석 종료일 문자열 또는 None
    change_method: str - 변화율 계산 방식 문자열("logdiff" 또는 "pct")
    scale: float - 변화율에 곱할 배율 값(보통 100.0)
    exclude_ranges: Optional[Sequence] - 분석에서 제외할 날짜 구간 목록 또는 None
    freq: str - 날짜 빈도 문자열(예: "D", "W", "M")
    fill_method: str - 결측 처리 방식 문자열("ffill" 또는 "none")
    """
    s = _ensure_datetime_index(data)

    if start is not None or end is not None:
        s = s.loc[start:end]

    s = s[[dom_col, intl_col]].rename(columns={dom_col: "dom_level", intl_col: "intl_level"})
    s = s.apply(pd.to_numeric, errors="coerce")

    if freq is not None:
        s = s.asfreq(freq)

    if fill_method == "ffill":
        s[["dom_level", "intl_level"]] = s[["dom_level", "intl_level"]].ffill()
    elif fill_method == "none":
        pass
    else:
        raise ValueError("fill_method 는 'ffill' 또는 'none' 여야 합니다.")

    excluded = pd.Series(False, index=s.index, name="excluded")
    for a, b in normalize_exclude_ranges(exclude_ranges):
        excluded |= (s.index >= a) & (s.index <= b)

    s.loc[excluded, ["dom_level", "intl_level"]] = np.nan

    if change_method == "logdiff":
        chk = s[["dom_level", "intl_level"]].dropna()
        if (chk <= 0).any().any():
            raise ValueError("change_method='logdiff' 인데 0 이하 값이 있습니다.")
        s["dom_chg"] = np.log(s["dom_level"]).diff() * scale
        s["intl_chg"] = np.log(s["intl_level"]).diff() * scale
    elif change_method == "pct":
        s["dom_chg"] = (s["dom_level"] / s["dom_level"].shift(1) - 1.0) * scale
        s["intl_chg"] = (s["intl_level"] / s["intl_level"].shift(1) - 1.0) * scale
    else:
        raise ValueError("change_method 는 'logdiff' 또는 'pct' 여야 합니다.")

    s["excluded"] = excluded
    return s

In [ ]:
# =========================================================
# 2) stationarity 진단
# =========================================================
def stationarity_report(series: pd.Series, name: str) -> Dict[str, float]:
    """
    입력: series: pd.Series, name: str
    출력: Dict[str, float]
    설명: 시계열에 대해 ADF와 KPSS 검정을 수행하고 정상성 유사 여부를 요약
    """
    z = pd.Series(series).dropna()
    out: Dict[str, float] = {"series": name, "n_obs": int(len(z))}

    try:
        adf_res = adfuller(z, autolag="AIC")
        out["adf_stat"] = float(adf_res[0])
        out["adf_p"] = float(adf_res[1])
    except Exception as e:  # pragma: no cover - defensive
        out["adf_stat"] = np.nan
        out["adf_p"] = np.nan
        out["adf_error"] = str(e)

    try:
        kpss_res = kpss(z, regression="c", nlags="auto")
        out["kpss_stat"] = float(kpss_res[0])
        out["kpss_p"] = float(kpss_res[1])
    except Exception as e:  # pragma: no cover - defensive
        out["kpss_stat"] = np.nan
        out["kpss_p"] = np.nan
        out["kpss_error"] = str(e)

    out["stationary_like"] = bool(
        np.isfinite(out.get("adf_p", np.nan))
        and np.isfinite(out.get("kpss_p", np.nan))
        and (out["adf_p"] < 0.05)
        and (out["kpss_p"] > 0.05)
    )
    return out

In [ ]:
# =========================================================
# 3) 달력효과 더미
#    - daily 데이터면 weekday 더미를 기본 권장
# =========================================================

def build_calendar_fixed(
    index: pd.DatetimeIndex,
    add_weekday: bool = True,
    add_month: bool = False,
) -> pd.DataFrame:

    """
    입력: index: pd.DatetimeIndex, add_weekday: bool=True, add_month: bool=False
    출력: pd.DataFrame
    설명: 요일 더미와 월 더미를 생성해 회귀용 고정효과 변수로 반환
    """
    idx = pd.DatetimeIndex(index)
    parts: List[pd.DataFrame] = []

    if add_weekday:
        wd = pd.get_dummies(idx.dayofweek, prefix="wd", drop_first=True, dtype=float)
        wd.index = idx
        parts.append(wd)

    if add_month:
        mm = pd.get_dummies(idx.month, prefix="m", drop_first=True, dtype=float)
        mm.index = idx
        parts.append(mm)

    if not parts:
        return pd.DataFrame(index=idx)
    return pd.concat(parts, axis=1)

In [ ]:
# =========================================================
# 4) lag base 생성
# =========================================================

def build_lagged_base(
    change_df: pd.DataFrame,
    max_p: int,
    max_q: int,
    causal: bool = True,
    fixed_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:

    """
    입력: change_df: pd.DataFrame, max_p: int, max_q: int, causal: bool=True, fixed_df: Optional[pd.DataFrame]=None
    출력: pd.DataFrame
    설명: 국내 자기지연항, 국제 지연항, 이전 국내 레벨, 고정효과를 포함한 기본 lag 데이터 생성
    """

    base = change_df[["dom_level", "intl_level", "dom_chg", "intl_chg"]].copy()
    base["prev_dom_level"] = base["dom_level"].shift(1)

    for i in range(1, max_p + 1):
        base[f"dom_L{i}"] = base["dom_chg"].shift(i)

    start_q = 1 if causal else 0
    for j in range(start_q, max_q + 1):
        base[f"intl_L{j}"] = base["intl_chg"].shift(j)

    if fixed_df is not None and len(fixed_df.columns) > 0:
        base = base.join(fixed_df, how="left")

    return base



def all_candidate_lag_cols(max_p: int, max_q: int, causal: bool = True) -> List[str]:

    """
    입력: max_p: int, max_q: int, causal: bool=True
    출력: List[str]
    설명: 후보가 되는 모든 국내/국제 lag 변수명을 반환
    """

    cols = [f"dom_L{i}" for i in range(1, max_p + 1)]
    start_q = 1 if causal else 0
    cols += [f"intl_L{j}" for j in range(start_q, max_q + 1)]
    return cols



def candidate_feature_cols(
    p: int,
    q: int,
    causal: bool = True,
    fixed_cols: Optional[Sequence[str]] = None,
) -> List[str]:

    """
    입력: p: int, q: int, causal: bool=True, fixed_cols: Optional[Sequence[str]]=None
    출력: List[str]
    설명: 특정 p, q 모형에서 실제 사용할 설명변수 컬럼명 목록을 반환
    """

    cols = [f"dom_L{i}" for i in range(1, p + 1)]
    start_q = 1 if causal else 0
    cols += [f"intl_L{j}" for j in range(start_q, q + 1)]
    cols += list(fixed_cols or [])
    return cols

In [ ]:
# =========================================================
# 5) design matrix
#    - force_same_sample=True 는 ARDL hold_back와 같은 목적
# =========================================================

def build_design_df(
    base: pd.DataFrame,
    p: int,
    q: int,
    causal: bool = True,
    trend: str = "c",
    fixed_cols: Optional[Sequence[str]] = None,
    force_same_sample: bool = False,
    max_p: Optional[int] = None,
    max_q: Optional[int] = None,
) -> pd.DataFrame:

    """
    입력: base: pd.DataFrame, p: int, q: int, causal: bool=True, trend: str="c", fixed_cols: Optional[Sequence[str]]=None, force_same_sample: bool=False, max_p: Optional[int]=None, max_q: Optional[int]=None
    출력: pd.DataFrame
    설명: 선택된 lag 구조와 고정효과를 반영한 회귀용 설계행렬 생성
    """

    fixed_cols = list(fixed_cols or [])
    feat_cols = candidate_feature_cols(p, q, causal=causal, fixed_cols=fixed_cols)

    if force_same_sample:
        if max_p is None or max_q is None:
            raise ValueError("force_same_sample=True 이면 max_p/max_q 가 필요합니다.")
        use_cols = ["dom_chg", "dom_level", "prev_dom_level"] + all_candidate_lag_cols(
            max_p, max_q, causal=causal
        ) + fixed_cols
        df = base[use_cols].dropna().copy()
    else:
        use_cols = ["dom_chg", "dom_level", "prev_dom_level"] + feat_cols
        df = base[use_cols].dropna().copy()

    X = df[feat_cols].copy()
    if trend == "c":
        X = sm.add_constant(X, has_constant="add")
    elif trend == "n":
        pass
    else:
        raise NotImplementedError("trend 는 'c' 또는 'n' 만 구현했습니다.")

    return pd.concat([df[["dom_chg", "dom_level", "prev_dom_level"]], X], axis=1)

In [ ]:
# =========================================================
# 6) 안정성 / 진단
# =========================================================

def ar_roots_from_params(params: pd.Series, p: int) -> np.ndarray:

    """
    입력: params: pd.Series, p: int
    출력: np.ndarray
    설명: 국내 자기회귀 계수로부터 AR 다항식의 근을 계산
    """

    phi = np.array([params.get(f"dom_L{i}", 0.0) for i in range(1, p + 1)], dtype=float)
    if p == 0:
        return np.array([])
    poly = np.r_[-phi[::-1], 1.0]
    return np.roots(poly)



def ar_stable(params: pd.Series, p: int, tol: float = 1e-10) -> Tuple[bool, np.ndarray]:
    """
    입력: params: pd.Series, p: int, tol: float=1e-10
    출력: Tuple[bool, np.ndarray]
    설명: AR 근의 절댓값을 이용해 자기회귀 구조의 안정성 여부를 판정
    """

    roots = ar_roots_from_params(params, p)
    if len(roots) == 0:
        return True, roots
    return bool(np.all(np.abs(roots) > 1.0 + tol)), roots



def _choose_lb_lag(n_obs: int, p: int, base_lag: int = 14) -> int:
    """
    입력: n_obs: int, p: int, base_lag: int=14
    출력: int
    설명: Ljung-Box 검정에 사용할 적절한 lag 수를 자동 선택
    """

    max_reasonable = max(5, n_obs // 5)
    return int(min(max(base_lag, p + 5), max_reasonable))



def fit_dynamic_regression(
    design_df: pd.DataFrame,
    p: int,
    q: int,
    causal: bool = True,
    fixed_cols: Optional[Sequence[str]] = None,
    cov_type: str = "HAC",
    hac_lags: int = 7,
    lb_lag_base: int = 14,
) -> Tuple[sm.regression.linear_model.RegressionResultsWrapper, Dict[str, float]]:

    """
    입력: design_df: pd.DataFrame, p: int, q: int, causal: bool=True, fixed_cols: Optional[Sequence[str]]=None, cov_type: str="HAC", hac_lags: int=7, lb_lag_base: int=14
    출력: Tuple[RegressionResultsWrapper, Dict[str, float]]
    설명: 동적 회귀모형을 적합하고 안정성·국제변수 블록 유의성·잔차 자기상관을 진단
    """

    fixed_cols = list(fixed_cols or [])
    feature_cols = [
        c for c in design_df.columns if c == "const" or c.startswith("dom_L") or c.startswith("intl_L") or c in fixed_cols
    ]
    y = design_df["dom_chg"]
    X = design_df[feature_cols]

    if cov_type == "HAC":
        res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    else:
        res = sm.OLS(y, X).fit(cov_type=cov_type)

    stable, roots = ar_stable(res.params, p)

    intl_cols = [c for c in feature_cols if c.startswith("intl_L")]
    intl_block_p = np.nan
    if intl_cols:
        R = np.zeros((len(intl_cols), len(feature_cols)))
        for r, cname in enumerate(intl_cols):
            R[r, feature_cols.index(cname)] = 1.0
        try:
            intl_block_p = float(res.f_test(R).pvalue)
        except Exception:
            intl_block_p = np.nan

    lb_p = np.nan
    lb_lag = np.nan
    try:
        lb_lag = _choose_lb_lag(len(res.resid), p, base_lag=lb_lag_base)
        # AutoReg 문서의 관례대로 model_df 는 AR lag 수(p)로 조정한다.
        lb_df = acorr_ljungbox(
            res.resid,
            lags=[lb_lag],
            model_df=p,
            return_df=True,
        )
        lb_p = float(lb_df["lb_pvalue"].iloc[-1])
    except Exception:
        pass

    return res, {
        "stable": stable,
        "max_abs_root": float(np.max(np.abs(roots))) if len(roots) else np.nan,
        "min_abs_root": float(np.min(np.abs(roots))) if len(roots) else np.nan,
        "intl_block_p": intl_block_p,
        "lb_pvalue": lb_p,
        "lb_lag": lb_lag,
        "n_obs": int(len(design_df)),
        "n_params": int(len(res.params)),
    }

In [ ]:
# =========================================================
# 7) one-step 변화율 -> level 재구성
# =========================================================

def rebuild_level(prev_level: float, pred_change: float, change_method: str = "logdiff") -> float:
    """
    입력: prev_level: float, pred_change: float, change_method: str="logdiff"
    출력: float
    설명: 예측된 변화율을 이전 레벨에 적용해 현재 레벨 예측값으로 복원
    """

    if not np.isfinite(prev_level) or not np.isfinite(pred_change):
        return np.nan
    if change_method == "logdiff":
        return float(prev_level * np.exp(pred_change / 100.0))
    if change_method == "pct":
        return float(prev_level * (1.0 + pred_change / 100.0))
    raise ValueError("change_method 는 'logdiff' 또는 'pct' 여야 합니다.")

In [ ]:
# =========================================================
# 8) rolling-origin OOS 평가
# =========================================================

def rolling_origin_eval_change(
    base: pd.DataFrame,
    p: int,
    q: int,
    start_train: int = 365,
    causal: bool = True,
    trend: str = "c",
    fixed_cols: Optional[Sequence[str]] = None,
    cov_type: str = "HAC",
    hac_lags: int = 7,
    change_method: str = "logdiff",
    show_progress: bool = False,
    progress_desc: Optional[str] = None,
) -> Tuple[pd.DataFrame, Dict[str, float]]:

    """
    입력: base: pd.DataFrame, p: int, q: int, start_train: int=365, causal: bool=True, trend: str="c", fixed_cols: Optional[Sequence[str]]=None, cov_type: str="HAC", hac_lags: int=7, change_method: str="logdiff"
    출력: Tuple[pd.DataFrame, Dict[str, float]]
    설명: rolling-origin 방식으로 표본외 예측을 반복 수행하고 변화율·레벨 기준 성능을 계산
    """

    fixed_cols = list(fixed_cols or [])
    design_df = build_design_df(
        base,
        p,
        q,
        causal=causal,
        trend=trend,
        fixed_cols=fixed_cols,
        force_same_sample=False,
    )

    if len(design_df) <= start_train:
        raise ValueError(f"유효 표본 부족: {len(design_df)} <= start_train={start_train}")

    feature_cols = [
        c for c in design_df.columns if c == "const" or c.startswith("dom_L") or c.startswith("intl_L") or c in fixed_cols
    ]

    recs: List[Dict[str, float]] = []

    iterator = range(start_train, len(design_df))
    if show_progress:
        iterator = tqdm(
            iterator,
            total=len(design_df) - start_train,
            desc=progress_desc or f"OOS p={p}, q={q}",
            leave=False,
        )

    for origin in iterator:
        train = design_df.iloc[:origin]
        test = design_df.iloc[[origin]]

        y_train = train["dom_chg"]
        X_train = train[feature_cols]
        X_test = test[feature_cols]

        if cov_type == "HAC":
            res = sm.OLS(y_train, X_train).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
        else:
            res = sm.OLS(y_train, X_train).fit(cov_type=cov_type)

        pred_chg = float(res.predict(X_test).iloc[0])
        actual_chg = float(test["dom_chg"].iloc[0])
        prev_level = float(test["prev_dom_level"].iloc[0])
        actual_level = float(test["dom_level"].iloc[0])
        pred_level = rebuild_level(prev_level, pred_chg, change_method=change_method)

        recs.append(
            {
                "date": test.index[0],
                "actual_chg": actual_chg,
                "pred_chg": pred_chg,
                "prev_dom_level": prev_level,
                "actual_level": actual_level,
                "pred_level": pred_level,
            }
        )

    out = pd.DataFrame(recs).set_index("date")
    out["err_chg"] = out["actual_chg"] - out["pred_chg"]
    out["abs_err_chg"] = np.abs(out["err_chg"])
    out["sq_err_chg"] = out["err_chg"] ** 2
    out["err_level"] = out["actual_level"] - out["pred_level"]
    out["abs_err_level"] = np.abs(out["err_level"])
    out["sq_err_level"] = out["err_level"] ** 2
    out["ape_level_pct"] = (out["abs_err_level"] / out["actual_level"]).replace([np.inf, -np.inf], np.nan) * 100

    metrics = {
        "rmse_chg": float(np.sqrt(out["sq_err_chg"].mean())),
        "mae_chg": float(out["abs_err_chg"].mean()),
        "rmse_level": float(np.sqrt(out["sq_err_level"].mean())),
        "mae_level": float(out["abs_err_level"].mean()),
        "mape_level_pct": float(out["ape_level_pct"].dropna().mean()),
        "n_forecasts": int(len(out)),
    }
    return out, metrics

In [ ]:
# =========================================================
# 9) full-sample 적합
# =========================================================

def fit_final_change_model(
    base: pd.DataFrame,
    p: int,
    q: int,
    causal: bool = True,
    trend: str = "c",
    fixed_cols: Optional[Sequence[str]] = None,
    cov_type: str = "HAC",
    hac_lags: int = 7,
    change_method: str = "logdiff",
    lb_lag_base: int = 14,
) -> Tuple[
    sm.regression.linear_model.RegressionResultsWrapper,
    Dict[str, float],
    pd.DataFrame,
    pd.DataFrame,
]:
    """
    입력: base: pd.DataFrame, p: int, q: int, causal: bool=True, trend: str="c", fixed_cols: Optional[Sequence[str]]=None, cov_type: str="HAC", hac_lags: int=7, change_method: str="logdiff", lb_lag_base: int=14
    출력: Tuple[RegressionResultsWrapper, Dict[str, float], pd.DataFrame, pd.DataFrame]
    설명: 최종 선택된 p, q로 전체 표본 회귀를 적합하고 적합값 및 오차 테이블을 생성
    """

    fixed_cols = list(fixed_cols or [])
    design_df = build_design_df(
        base,
        p,
        q,
        causal=causal,
        trend=trend,
        fixed_cols=fixed_cols,
        force_same_sample=False,
    )
    res, diag = fit_dynamic_regression(
        design_df,
        p,
        q,
        causal=causal,
        fixed_cols=fixed_cols,
        cov_type=cov_type,
        hac_lags=hac_lags,
        lb_lag_base=lb_lag_base,
    )

    feature_cols = [
        c for c in design_df.columns if c == "const" or c.startswith("dom_L") or c.startswith("intl_L") or c in fixed_cols
    ]
    fit_df = design_df[["dom_chg", "dom_level", "prev_dom_level"]].copy()
    fit_df["fitted_chg"] = res.predict(design_df[feature_cols])
    fit_df["fitted_level"] = [
        rebuild_level(prev, pred, change_method=change_method)
        for prev, pred in zip(fit_df["prev_dom_level"], fit_df["fitted_chg"])
    ]
    fit_df["err_chg"] = fit_df["dom_chg"] - fit_df["fitted_chg"]
    fit_df["err_level"] = fit_df["dom_level"] - fit_df["fitted_level"]

    return res, diag, design_df, fit_df

In [ ]:
# =========================================================
# 10) impulse response 기반 대표 지연값
# =========================================================

def impulse_response_stats(
    res: sm.regression.linear_model.RegressionResultsWrapper,
    p: int,
    q: int,
    horizon: int = 60,
    shock: float = 1.0,
    causal: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:

    """
    입력: res: RegressionResultsWrapper, p: int, q: int, horizon: int=60, shock: float=1.0, causal: bool=True
    출력: Tuple[pd.DataFrame, Dict[str, float]]
    설명: 국제 가격 충격에 대한 국내 변화율 반응경로와 대표 지연 통계를 계산
    """

    phi = {i: float(res.params.get(f"dom_L{i}", 0.0)) for i in range(1, p + 1)}
    start_q = 1 if causal else 0
    beta = {j: float(res.params.get(f"intl_L{j}", 0.0)) for j in range(start_q, q + 1)}

    resp = np.zeros(horizon + 1, dtype=float)
    for h in range(horizon + 1):
        y_part = sum(phi.get(i, 0.0) * resp[h - i] for i in range(1, p + 1) if h - i >= 0)
        x_part = beta.get(h, 0.0) * shock
        resp[h] = y_part + x_part

    absw = np.abs(resp)
    mean_lag = np.nan
    half_mass_lag = np.nan
    if absw.sum() > 1e-12:
        mean_lag = float(np.sum(np.arange(len(resp)) * absw) / absw.sum())
        cum_abs = np.cumsum(absw)
        half_mass_lag = int(np.where(cum_abs >= 0.5 * cum_abs[-1])[0][0])

    path = pd.DataFrame(
        {
            "h": np.arange(horizon + 1),
            "impulse_response": resp,
            "cumulative_response": np.cumsum(resp),
        }
    )
    stats = {
        "peak_lag": int(np.argmax(np.abs(resp))),
        "mean_lag": mean_lag,
        "half_mass_lag": half_mass_lag,
        "cum_response_horizon": float(path["cumulative_response"].iloc[-1]),
    }
    return path, stats

In [ ]:
# =========================================================
# 11) 년도별 요약
# =========================================================

def summarize_oos_by_year_change(oos_df: pd.DataFrame) -> pd.DataFrame:
    """
    함수명: summarize_oos_by_year_change
    입력: oos_df: pd.DataFrame
    출력: pd.DataFrame
    설명: 표본외 예측 결과를 연도별 오차 및 평균 지표로 요약
    """

    t = oos_df.copy()
    t["year"] = t.index.year
    return (
        t.groupby("year")
        .agg(
            n_obs=("err_chg", "size"),
            actual_chg_mean=("actual_chg", "mean"),
            pred_chg_mean=("pred_chg", "mean"),
            bias_chg=("err_chg", "mean"),
            mae_chg=("abs_err_chg", "mean"),
            rmse_chg=("sq_err_chg", lambda s: float(np.sqrt(np.mean(s)))),
            actual_level_mean=("actual_level", "mean"),
            pred_level_mean=("pred_level", "mean"),
            bias_level=("err_level", "mean"),
            mae_level=("abs_err_level", "mean"),
            rmse_level=("sq_err_level", lambda s: float(np.sqrt(np.mean(s)))),
            mape_level_pct=("ape_level_pct", "mean"),
        )
        .reset_index()
    )



def summarize_fit_by_year_change(fit_df: pd.DataFrame) -> pd.DataFrame:
    """
    입력: fit_df: pd.DataFrame
    출력: pd.DataFrame
    설명: 전체표본 적합 결과를 연도별 오차 및 평균 지표로 요약
    """

    t = fit_df.copy()
    t["year"] = t.index.year
    t["abs_err_chg"] = np.abs(t["err_chg"])
    t["sq_err_chg"] = t["err_chg"] ** 2
    t["abs_err_level"] = np.abs(t["err_level"])
    t["sq_err_level"] = t["err_level"] ** 2
    t["ape_level_pct"] = (t["abs_err_level"] / t["dom_level"]).replace([np.inf, -np.inf], np.nan) * 100
    return (
        t.groupby("year")
        .agg(
            n_obs=("err_chg", "size"),
            actual_chg_mean=("dom_chg", "mean"),
            fitted_chg_mean=("fitted_chg", "mean"),
            bias_chg=("err_chg", "mean"),
            mae_chg=("abs_err_chg", "mean"),
            rmse_chg=("sq_err_chg", lambda s: float(np.sqrt(np.mean(s)))),
            actual_level_mean=("dom_level", "mean"),
            fitted_level_mean=("fitted_level", "mean"),
            bias_level=("err_level", "mean"),
            mae_level=("abs_err_level", "mean"),
            rmse_level=("sq_err_level", lambda s: float(np.sqrt(np.mean(s)))),
            mape_level_pct=("ape_level_pct", "mean"),
        )
        .reset_index()
    )

In [ ]:
# =========================================================
# 12) lag selection
#    - 1차: train-sample BIC + 안정성 + intl block + Ljung-Box
#    - 2차: 상위 후보들 rolling OOS 비교
#    - 3차: full-sample Ljung-Box 재점검
# =========================================================

def select_orders_bic_change_v2(
    base: pd.DataFrame,
    max_p: int = 31,
    max_q: int = 31,
    min_train: int = 365,
    causal: bool = True,
    trend: str = "c",
    fixed_cols: Optional[Sequence[str]] = None,
    cov_type: str = "HAC",
    hac_lags: int = 7,
    lb_alpha: float = 0.05,
    intl_block_alpha: float = 0.10,
    lb_lag_base: int = 14,
    top_n_bic: int = 12,
    change_method: str = "logdiff",
    show_progress: bool = False,
) -> Tuple[Dict[str, float], pd.DataFrame, pd.DataFrame]:
    """
    입력: base: pd.DataFrame, max_p: int=31, max_q: int=31, min_train: int=365, causal: bool=True, trend: str="c", fixed_cols: Optional[Sequence[str]]=None, cov_type: str="HAC", hac_lags: int=7, lb_alpha: float=0.05, intl_block_alpha: float=0.10, lb_lag_base: int=14, top_n_bic: int=12, change_method: str="logdiff"
    출력: Tuple[Dict[str, float], pd.DataFrame, pd.DataFrame]
    설명: 후보 p, q 조합을 BIC·안정성·잔차진단·국제변수 유의성·OOS 성능 기준으로 비교해 최적 차수를 선택
    """

    fixed_cols = list(fixed_cols or [])
    rows: List[Dict[str, float]] = []

    q_start = 1 if causal else 0
    grid_iter = (
        (p, q)
        for p in range(0, max_p + 1)
        for q in range(q_start, max_q + 1)
    )

    if show_progress:
        grid_iter = tqdm(
            grid_iter,
            total=(max_p + 1) * (max_q - q_start + 1),
            desc="Grid search (p,q)",
        )

    for p, q in grid_iter:
        try:
            design_df = build_design_df(
                base,
                p,
                q,
                causal=causal,
                trend=trend,
                fixed_cols=fixed_cols,
                force_same_sample=True,
                max_p=max_p,
                max_q=max_q,
            )
            if len(design_df) < min_train:
                raise ValueError(f"유효 표본 부족: {len(design_df)} < {min_train}")
            design_train = design_df.iloc[:min_train].copy()
            res, diag = fit_dynamic_regression(
                design_train,
                p,
                q,
                causal=causal,
                fixed_cols=fixed_cols,
                cov_type=cov_type,
                hac_lags=hac_lags,
                lb_lag_base=lb_lag_base,
            )
            rows.append(
                {
                    "p": p,
                    "q": q,
                    "aic": float(res.aic),
                    "bic": float(res.bic),
                    "stable": bool(diag["stable"]),
                    "intl_block_p": diag["intl_block_p"],
                    "lb_pvalue": diag["lb_pvalue"],
                    "lb_lag": diag["lb_lag"],
                    "n_obs": diag["n_obs"],
                }
            )
        except Exception as e:
            rows.append(
                {
                    "p": p,
                    "q": q,
                    "aic": np.nan,
                    "bic": np.nan,
                    "stable": False,
                    "intl_block_p": np.nan,
                    "lb_pvalue": np.nan,
                    "lb_lag": np.nan,
                    "n_obs": np.nan,
                    "error": str(e),
                }
            )

    grid = pd.DataFrame(rows)
    ok = grid["bic"].notna()
    grid["train_lb_pass"] = grid["lb_pvalue"] >= lb_alpha
    grid["train_intl_pass"] = grid["intl_block_p"] <= intl_block_alpha

    eligible = grid.loc[ok & grid["stable"] & grid["train_lb_pass"] & grid["train_intl_pass"]].copy()
    selection_rule = "stable + train_lb_pass + intl_block_pass"
    if eligible.empty:
        eligible = grid.loc[ok & grid["stable"] & grid["train_intl_pass"]].copy()
        selection_rule = "stable + intl_block_pass"
    if eligible.empty:
        eligible = grid.loc[ok & grid["stable"]].copy()
        selection_rule = "stable_only"
    if eligible.empty:
        eligible = grid.loc[ok].copy()
        selection_rule = "bic_only"

    shortlist_bic = (
    eligible.sort_values(["bic", "q", "p"], na_position="last")
    .head(top_n_bic)
    .copy()
    )

    shortlist_lb = (
        eligible.sort_values(
            ["lb_pvalue", "bic", "q", "p"],
            ascending=[False, True, True, True],
            na_position="last",
        )
        .head(top_n_bic)
        .copy()
    )

    shortlist = (
        pd.concat([shortlist_bic, shortlist_lb], axis=0)
        .drop_duplicates(subset=["p", "q"])
        .reset_index(drop=True)
    )

    finalists: List[Dict[str, float]] = []

    finalist_iter = shortlist.iterrows()
    if show_progress:
        finalist_iter = tqdm(
            finalist_iter,
            total=len(shortlist),
            desc="Finalist OOS/full fit",
            leave=False,
        )

    for _, row in finalist_iter:
        p = int(row["p"])
        q = int(row["q"])
        try:
            oos_df, oos_metrics = rolling_origin_eval_change(
                base,
                p=p,
                q=q,
                start_train=min_train,
                causal=causal,
                trend=trend,
                fixed_cols=fixed_cols,
                cov_type=cov_type,
                hac_lags=hac_lags,
                change_method=change_method,
                show_progress=False,
            )
            final_res, final_diag, final_design, _ = fit_final_change_model(
                base,
                p=p,
                q=q,
                causal=causal,
                trend=trend,
                fixed_cols=fixed_cols,
                cov_type=cov_type,
                hac_lags=hac_lags,
                change_method=change_method,
                lb_lag_base=lb_lag_base,
            )
            finalists.append(
                {
                    **row.to_dict(),
                    "oos_rmse_chg": oos_metrics["rmse_chg"],
                    "oos_mae_chg": oos_metrics["mae_chg"],
                    "oos_rmse_level": oos_metrics["rmse_level"],
                    "oos_mae_level": oos_metrics["mae_level"],
                    "oos_mape_level_pct": oos_metrics["mape_level_pct"],
                    "full_lb_pvalue": final_diag["lb_pvalue"],
                    "full_lb_lag": final_diag["lb_lag"],
                    "full_stable": final_diag["stable"],
                    "full_intl_block_p": final_diag["intl_block_p"],
                    "full_n_obs": len(final_design),
                }
            )
        except Exception as e:
            finalists.append({**row.to_dict(), "oos_error": str(e)})

    finalists_df = pd.DataFrame(finalists)
    finalists_df["full_lb_pass"] = finalists_df.get("full_lb_pvalue", np.nan) >= lb_alpha

    chooser = finalists_df.copy()

    if "oos_rmse_chg" not in chooser.columns:
        chooser["oos_rmse_chg"] = np.nan

    chooser = chooser[pd.to_numeric(chooser["oos_rmse_chg"], errors="coerce").notna()].copy()

    if chooser.empty:
        best = shortlist.sort_values(["bic", "q", "p"], na_position="last").iloc[0].to_dict()
        best["selection_rule"] = selection_rule + " -> fallback shortlist BIC because all OOS failed"

        return (
            best,
            grid.sort_values(["bic", "p", "q"], na_position="last").reset_index(drop=True),
            finalists_df,
        )

    # ---------------------------------------------------------
    # 최종 선택: 빈 후보 방어 포함
    # ---------------------------------------------------------
    if "full_stable" not in chooser.columns:
        chooser["full_stable"] = False

    if "full_lb_pass" not in chooser.columns:
        chooser["full_lb_pass"] = False

    if "full_lb_pvalue" not in chooser.columns:
        chooser["full_lb_pvalue"] = np.nan

    strict = chooser.loc[
        chooser["full_stable"].fillna(False)
        & chooser["full_lb_pass"].fillna(False)
    ].copy()

    if not strict.empty:
        final_rule = selection_rule + " -> OOS rank + full_lb_pass"
        best = (
            strict.sort_values(
                ["oos_rmse_chg", "bic", "q", "p"],
                na_position="last",
            )
            .iloc[0]
            .to_dict()
        )

    else:
        relaxed = chooser.loc[
            chooser["full_stable"].fillna(False)
        ].copy()

        if not relaxed.empty:
            final_rule = selection_rule + " -> OOS rank among full_stable"
            best = (
                relaxed.sort_values(
                    ["oos_rmse_chg", "bic", "q", "p"],
                    na_position="last",
                )
                .iloc[0]
                .to_dict()
            )

        else:
            # 안정성 조건을 만족하는 finalist가 하나도 없을 때의 최후 fallback
            final_rule = selection_rule + " -> fallback best OOS without full_stable"
            best = (
                chooser.sort_values(
                    ["oos_rmse_chg", "full_lb_pvalue", "bic", "q", "p"],
                    ascending=[True, False, True, True, True],
                    na_position="last",
                )
                .iloc[0]
                .to_dict()
            )

    best["selection_rule"] = final_rule

    return (
        best,
        grid.sort_values(["bic", "p", "q"], na_position="last").reset_index(drop=True),
        finalists_df.sort_values(
            ["oos_rmse_chg", "bic", "p", "q"],
            na_position="last",
        ).reset_index(drop=True),
    )
    best["selection_rule"] = final_rule
    return best, grid.sort_values(["bic", "p", "q"], na_position="last").reset_index(drop=True), finalists_df.sort_values(["oos_rmse_chg", "bic", "p", "q"], na_position="last").reset_index(drop=True)

In [ ]:
# =========================================================
# 13) 전체 실행 함수
# =========================================================

def run_lag_analysis(
    data: pd.DataFrame,
    dom_col: str,
    intl_col: str,
    *,
    start: Optional[str] = None,
    end: Optional[str] = None,
    change_method: str = "logdiff",
    scale: float = 100.0,
    exclude_ranges: Optional[Sequence] = None,
    freq: str = "D",
    fill_method: str = "ffill",
    max_p: int = 31,
    max_q: int = 31,
    min_train: int = 3 * 365,
    response_h: int = 90,
    trend: str = "c",
    causal: bool = True,
    cov_type: str = "HAC",
    hac_lags: int = 7,
    lb_alpha: float = 0.05,
    intl_block_alpha: float = 0.10,
    lb_lag_base: int = 14,
    add_weekday_dummies: bool = True,
    add_month_dummies: bool = False,
    extra_fixed: Optional[pd.DataFrame] = None,
    top_n_bic: int = 12,
    show_progress: bool = False,
) -> Dict[str, object]:
    """
    입력: data: pd.DataFrame, dom_col: str, intl_col: str, start: Optional[str]=None, end: Optional[str]=None, change_method: str="logdiff", scale: float=100.0, exclude_ranges: Optional[Sequence]=None, freq: str="D", fill_method: str="ffill", max_p: int=31, max_q: int=31, min_train: int=3*365, response_h: int=90, trend: str="c", causal: bool=True, cov_type: str="HAC", hac_lags: int=7, lb_alpha: float=0.05, intl_block_alpha: float=0.10, lb_lag_base: int=14, add_weekday_dummies: bool=True, add_month_dummies: bool=False, extra_fixed: Optional[pd.DataFrame]=None, top_n_bic: int=12
    출력: Dict[str, object]
    설명: 전처리부터 차수선택·최종적합·표본외평가·충격반응 분석까지 전체 파이프라인을 한 번에 수행
    """

    if show_progress:
        print("[1/6] 변화율 데이터 생성")
    change_df = build_change_frame(
        data=data,
        dom_col=dom_col,
        intl_col=intl_col,
        start=start,
        end=end,
        change_method=change_method,
        scale=scale,
        exclude_ranges=exclude_ranges,
        freq=freq,
        fill_method=fill_method,
    )

    if show_progress:
        print("[2/6] 정상성 점검")
    dom_diag = stationarity_report(change_df["dom_chg"], "dom_chg")
    intl_diag = stationarity_report(change_df["intl_chg"], "intl_chg")
    stationarity_df = pd.DataFrame([dom_diag, intl_diag])

    if show_progress:
        print("[3/6] 고정효과 / lag base 생성")
    fixed_df = build_calendar_fixed(
        change_df.index,
        add_weekday=add_weekday_dummies if freq == "D" else False,
        add_month=add_month_dummies,
    )
    if extra_fixed is not None and len(extra_fixed.columns) > 0:
        extra = extra_fixed.copy()
        extra.index = pd.to_datetime(extra.index)
        fixed_df = fixed_df.join(extra, how="left") if len(fixed_df.columns) > 0 else extra

    fixed_cols = list(fixed_df.columns)
    base = build_lagged_base(change_df, max_p=max_p, max_q=max_q, causal=causal, fixed_df=fixed_df)

    if show_progress:
        print("[4/6] lag 후보 탐색")
    best_orders, order_grid, finalists = select_orders_bic_change_v2(
        base,
        max_p=max_p,
        max_q=max_q,
        min_train=min_train,
        causal=causal,
        trend=trend,
        fixed_cols=fixed_cols,
        cov_type=cov_type,
        hac_lags=hac_lags,
        lb_alpha=lb_alpha,
        intl_block_alpha=intl_block_alpha,
        lb_lag_base=lb_lag_base,
        top_n_bic=top_n_bic,
        change_method=change_method,
        show_progress=show_progress,
    )

    p_best = int(best_orders["p"])
    q_best = int(best_orders["q"])


    if show_progress:
        print(f"[4/6] 완료 -> selected p={p_best}, q={q_best}")
        print("[5/6] 선택 모형 rolling OOS")
    oos_df, oos_metrics = rolling_origin_eval_change(
        base,
        p=p_best,
        q=q_best,
        start_train=min_train,
        causal=causal,
        trend=trend,
        fixed_cols=fixed_cols,
        cov_type=cov_type,
        hac_lags=hac_lags,
        change_method=change_method,
        show_progress=show_progress,
        progress_desc=f"Selected model OOS p={p_best}, q={q_best}",
    )

    if show_progress:
        print("[6/6] 최종 적합 / 충격반응 / 요약")
    final_res, final_diag, final_design, fit_df = fit_final_change_model(
        base,
        p=p_best,
        q=q_best,
        causal=causal,
        trend=trend,
        fixed_cols=fixed_cols,
        cov_type=cov_type,
        hac_lags=hac_lags,
        change_method=change_method,
        lb_lag_base=lb_lag_base,
    )

    irf_path, irf_stats = impulse_response_stats(
        final_res,
        p=p_best,
        q=q_best,
        horizon=response_h,
        shock=1.0,
        causal=causal,
    )

    shift_days = 0
    if np.isfinite(irf_stats["mean_lag"]):
        shift_days = max(0, int(round(irf_stats["mean_lag"])))

    summary = pd.Series(
        {
            "n_raw_days": len(change_df),
            "n_effective_final": len(final_design),
            "excluded_days": int(change_df["excluded"].sum()),
            "change_method": change_method,
            "freq": freq,
            "fill_method": fill_method,
            "causal_only": causal,
            "selected_p": p_best,
            "selected_q": q_best,
            "selection_rule": best_orders.get("selection_rule", ""),
            "selection_bic": float(best_orders.get("bic", np.nan)),
            "selection_train_lb_pvalue": float(best_orders.get("lb_pvalue", np.nan)),
            "selection_full_lb_pvalue": float(final_diag.get("lb_pvalue", np.nan)),
            "selection_train_intl_block_p": float(best_orders.get("intl_block_p", np.nan)),
            "selection_full_intl_block_p": float(final_diag.get("intl_block_p", np.nan)),
            "dom_chg_adf_p": dom_diag.get("adf_p", np.nan),
            "dom_chg_kpss_p": dom_diag.get("kpss_p", np.nan),
            "intl_chg_adf_p": intl_diag.get("adf_p", np.nan),
            "intl_chg_kpss_p": intl_diag.get("kpss_p", np.nan),
            "change_model_ok": bool(dom_diag["stationary_like"] and intl_diag["stationary_like"]),
            "final_stable": bool(final_diag["stable"]),
            "oos_rmse_chg": oos_metrics["rmse_chg"],
            "oos_mae_chg": oos_metrics["mae_chg"],
            "oos_rmse_level": oos_metrics["rmse_level"],
            "oos_mae_level": oos_metrics["mae_level"],
            "oos_mape_level_pct": oos_metrics["mape_level_pct"],
            "irf_peak_lag": irf_stats["peak_lag"],
            "irf_half_mass_lag": irf_stats["half_mass_lag"],
            "irf_mean_lag": irf_stats["mean_lag"],
            "irf_cum_response_horizon": irf_stats["cum_response_horizon"],
            "recommended_shift_days": shift_days,
            "n_fixed_regressors": len(fixed_cols),
        }
    )

    return {
        "summary": summary,
        "stationarity": stationarity_df,
        "change_df": change_df,
        "fixed_df": fixed_df,
        "order_grid": order_grid,
        "finalists": finalists,
        "oos_df": oos_df,
        "oos_metrics": oos_metrics,
        "oos_by_year": summarize_oos_by_year_change(oos_df),
        "final_res": final_res,
        "final_diag": final_diag,
        "final_design": final_design,
        "fit_df": fit_df,
        "fit_by_year": summarize_fit_by_year_change(fit_df),
        "irf_path": irf_path,
        "irf_stats": irf_stats,
    }

In [ ]:
# =========================================================
# 14) 결과 저장 함수
# =========================================================

def _ensure_path(path_like) -> Path:
    return path_like if isinstance(path_like, Path) else Path(path_like)

def make_lag_analysis_output_dir(root_path, result_name: str):
    root_dir = _ensure_path(root_path)
    lag_analysis_dir = root_dir / f"{result_file_name}"
    result_dir = lag_analysis_dir / result_name

    lag_analysis_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    return {
        "root_dir": root_dir,
        "lag_analysis_dir": lag_analysis_dir,
        "result_dir": result_dir,
    }


def _save_series_as_one_row_csv(series, path):
    series.to_frame().T.to_csv(path, index=False, encoding="utf-8-sig")

def _save_series_as_json(series, path):
    payload = {}
    for k, v in series.to_dict().items():
        if isinstance(v, (np.integer, np.int64)):
            payload[k] = int(v)
        elif isinstance(v, (np.floating, np.float64, float)):
            payload[k] = None if not np.isfinite(v) else float(v)
        elif isinstance(v, (np.bool_, bool)):
            payload[k] = bool(v)
        else:
            payload[k] = v
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def _save_model_params(final_res, path):
    out = pd.DataFrame({
        "param": final_res.params.index,
        "coef": final_res.params.values,
        "std_err": final_res.bse.reindex(final_res.params.index).values,
        "t_value": final_res.tvalues.reindex(final_res.params.index).values,
        "p_value": final_res.pvalues.reindex(final_res.params.index).values,
    })
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return out

def _save_text(text, path):
    path.write_text(text, encoding="utf-8")


def save_lag_analysis_results(
    result,
    root_path,
    result_name: str,
    *,
    save_change_frame: bool = False,
    save_fixed_frame: bool = False,
    plot_title_prefix: Optional[str] = None,
):
    dirs = make_lag_analysis_output_dir(root_path=root_path, result_name=result_name)
    result_dir = dirs["result_dir"]

    _save_series_as_one_row_csv(result["summary"], result_dir / "analysis_summary.csv")
    _save_series_as_json(result["summary"], result_dir / "analysis_summary.json")

    result["stationarity"].to_csv(result_dir / "stationarity_tests.csv", index=False, encoding="utf-8-sig")
    result["order_grid"].to_csv(result_dir / "candidate_grid.csv", index=False, encoding="utf-8-sig")
    result["finalists"].to_csv(result_dir / "finalist_models.csv", index=False, encoding="utf-8-sig")
    result["oos_df"].to_csv(result_dir / "oos_predictions.csv", encoding="utf-8-sig")
    pd.DataFrame([result["oos_metrics"]]).to_csv(result_dir / "oos_metrics.csv", index=False, encoding="utf-8-sig")
    result["oos_by_year"].to_csv(result_dir / "oos_by_year.csv", index=False, encoding="utf-8-sig")
    result["fit_df"].to_csv(result_dir / "in_sample_fit.csv", encoding="utf-8-sig")
    result["fit_by_year"].to_csv(result_dir / "fit_by_year.csv", index=False, encoding="utf-8-sig")
    result["irf_path"].to_csv(result_dir / "impulse_response_path.csv", index=False, encoding="utf-8-sig")
    _save_model_params(result["final_res"], result_dir / "final_model_params.csv")
    _save_text(str(result["final_res"].summary()), result_dir / "final_model_summary.txt")

    if save_change_frame:
        result["change_df"].to_csv(result_dir / "change_frame.csv", encoding="utf-8-sig")
    if save_fixed_frame and len(result["fixed_df"].columns) > 0:
        result["fixed_df"].to_csv(result_dir / "fixed_regressors.csv", encoding="utf-8-sig")

    return {
        **dirs,
        "analysis_summary_csv": result_dir / "analysis_summary.csv",
        "analysis_summary_json": result_dir / "analysis_summary.json",
    }

def run_and_save_lag_analysis(
    data,
    dom_col,
    intl_col,
    *,
    root_path,
    result_name: str,
    start=None,
    end=None,
    change_method="logdiff",
    scale=100.0,
    exclude_ranges=None,
    freq="D",
    fill_method="ffill",
    max_p=31,
    max_q=31,
    min_train=3 * 365,
    response_h=90,
    trend="c",
    causal=True,
    cov_type="HAC",
    hac_lags=7,
    lb_alpha=0.05,
    intl_block_alpha=0.10,
    lb_lag_base=14,
    add_weekday_dummies=True,
    add_month_dummies=False,
    extra_fixed=None,
    top_n_bic=12,
    save_change_frame=False,
    save_fixed_frame=False,
    plot_title_prefix=None,
    show_progress: bool = False,
):
    result = run_lag_analysis(
        data=data,
        dom_col=dom_col,
        intl_col=intl_col,
        start=start,
        end=end,
        change_method=change_method,
        scale=scale,
        exclude_ranges=exclude_ranges,
        freq=freq,
        fill_method=fill_method,
        max_p=max_p,
        max_q=max_q,
        min_train=min_train,
        response_h=response_h,
        trend=trend,
        causal=causal,
        cov_type=cov_type,
        hac_lags=hac_lags,
        lb_alpha=lb_alpha,
        intl_block_alpha=intl_block_alpha,
        lb_lag_base=lb_lag_base,
        add_weekday_dummies=add_weekday_dummies,
        add_month_dummies=add_month_dummies,
        extra_fixed=extra_fixed,
        top_n_bic=top_n_bic,
        show_progress=show_progress,
    )
    saved_paths = save_lag_analysis_results(
        result=result,
        root_path=root_path,
        result_name=result_name,
        save_change_frame=save_change_frame,
        save_fixed_frame=save_fixed_frame,
        plot_title_prefix=plot_title_prefix,
    )
    result["saved_paths"] = saved_paths
    return result

In [ ]:
# =========================================================
# 1단계 분석 실행 함수
# - 주유소 일별 시차
# - 정유사 주간 시차
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT_DIR = Path(ROOT_PATH)

STAGE1_RESULTS = {}


def stage1_summary_row_safe(result_obj, stage_name: str, shift_unit_name: str) -> dict:
    s = result_obj["summary"].copy()

    def get_float(key):
        try:
            return float(s.get(key, np.nan))
        except Exception:
            return np.nan

    def get_int(key):
        try:
            return int(float(s.get(key, np.nan)))
        except Exception:
            return np.nan

    shift_key = "recommended_shift_days"
    if "recommended_shift_weeks" in s.index:
        shift_key = "recommended_shift_weeks"
    elif "recommended_shift_periods" in s.index:
        shift_key = "recommended_shift_periods"

    return {
        "stage": stage_name,
        "selected_p": get_int("selected_p"),
        "selected_q": get_int("selected_q"),
        "oos_rmse_chg": get_float("oos_rmse_chg"),
        "oos_rmse_level": get_float("oos_rmse_level"),
        "oos_mae_level": get_float("oos_mae_level"),
        "oos_mape_level_pct": get_float("oos_mape_level_pct"),
        "selection_train_lb_pvalue": get_float("selection_train_lb_pvalue"),
        "selection_full_lb_pvalue": get_float("selection_full_lb_pvalue"),
        "selection_full_intl_block_p": get_float("selection_full_intl_block_p"),
        "irf_peak_lag": get_float("irf_peak_lag"),
        "irf_half_mass_lag": get_float("irf_half_mass_lag"),
        "irf_mean_lag": get_float("irf_mean_lag"),
        "recommended_shift": get_int(shift_key),
        "recommended_shift_unit": shift_unit_name,
        "saved_dir": str(result_obj["saved_paths"]["result_dir"]),
    }


def expand_exclude_ranges_for_weekly(
    exclude_ranges,
    *,
    week_rule: str = "W-SAT",
    buffer_weeks: int = 1,
):
    out = []

    for item in exclude_ranges:
        if isinstance(item, dict):
            start = item["start"]
            end = item["end"]
            label = item.get("label", "")
        else:
            start, end = item
            label = ""

        s = pd.Timestamp(start).normalize() - pd.Timedelta(weeks=buffer_weeks)
        e = pd.Timestamp(end).normalize() + pd.Timedelta(weeks=buffer_weeks)

        s_week = s.to_period(week_rule).end_time.normalize()
        e_week = e.to_period(week_rule).end_time.normalize()

        out.append({
            "start": str(s_week.date()),
            "end": str(e_week.date()),
            "label": label,
        })

    return out


def run_consumer_daily_lag(
    fuel_key: str,
    benchmark_col: str | None = None,
    *,
    show_frames: bool = False,
):
    """
    국제제품가격 daily -> 주유소 소비자가격 daily 시차 분석.

    저장 폴더:
      시차분석/gasoline_lag_analysis
      시차분석/diesel_lag_analysis
    """
    frames = prepare_stage1_frames(
        fuel_key=fuel_key,
        benchmark_col=benchmark_col,
        verbose=show_frames,
    )

    fuel_label = frames["fuel_label"]

    exclude_ranges_consumer = EXCLUDE_RANGES_STAGE1

    main_daily_gross_params = dict(
        start=ANALYSIS_START,
        end=ANALYSIS_END,
        change_method="logdiff",
        exclude_ranges=exclude_ranges_consumer,
        freq="D",
        fill_method="ffill",
        max_p=60,
        max_q=60,
        min_train=3 * 365,
        response_h=90,
        causal=True,
        cov_type="HAC",
        hac_lags=7,
        lb_alpha=0.05,
        intl_block_alpha=0.10,
        lb_lag_base=14,
        add_weekday_dummies=True,
        add_month_dummies=True,
        top_n_bic=20,
        save_change_frame=False,
        save_fixed_frame=True,
        show_progress=True,
    )

    main_gross_input_df = frames["consumer_daily_gross_df"].rename(columns={
        "benchmark_daily": "intl_price",
        "retail_gross_daily": "dom_price",
    }).copy()

    result_obj = run_and_save_lag_analysis(
        data=main_gross_input_df,
        dom_col="dom_price",
        intl_col="intl_price",
        root_path=ROOT_DIR,
        result_name=f"{fuel_key}_lag_analysis",
        **main_daily_gross_params,
    )

    result_obj["summary"].loc["fuel_key"] = fuel_key
    result_obj["summary"].loc["fuel_label"] = fuel_label
    result_obj["summary"].loc["benchmark_col"] = frames["benchmark_col"]
    result_obj["summary"].loc["target_layer"] = "consumer_retail_gross_daily"
    result_obj["summary"].loc["target_description"] = "국제제품가격 일별 -> 주유소 소비자가격 일별"


    # 메타정보를 추가한 summary를 다시 저장
    _save_series_as_one_row_csv(
        result_obj["summary"],
        result_obj["saved_paths"]["result_dir"] / "analysis_summary.csv",
    )

    _save_series_as_json(
        result_obj["summary"],
        result_obj["saved_paths"]["result_dir"] / "analysis_summary.json",
    )

    consumer_shift_days = int(result_obj["summary"]["recommended_shift_days"])

    output = {
        "fuel": fuel_key,
        "fuel_label": fuel_label,
        "benchmark_col": frames["benchmark_col"],
        "consumer_daily_gross_df": frames["consumer_daily_gross_df"],
        "consumer_daily_pretax_df": frames["consumer_daily_pretax_df"],
        "weekly_upstream_df": frames["weekly_upstream_df"],
        "weekly_downstream_df": frames["weekly_downstream_df"],
        "weekly_direct_df": frames["weekly_direct_df"],
        "daily_operating_df": frames["daily_operating_df"],
        "brand_weekly_df": frames["brand_weekly_df"],
        "main_consumer_gross_result": result_obj,
        "CONSUMER_SHIFT_DAYS": consumer_shift_days,
    }

    STAGE1_RESULTS[(fuel_key, "consumer_daily")] = output

    # 기존 코드 호환: 마지막 실행 결과를 전역 변수에도 남긴다.
    globals()["stage1_outputs"] = output
    globals()["result_obj"] = output

    summary_df = pd.DataFrame([
        stage1_summary_row_safe(
            result_obj,
            f"1-A {fuel_label} consumer gross daily",
            "days",
        )
    ])

    print("\n[1-A 주유소 소비자가격 일별 시차 분석 summary]")
    display(summary_df)

    print("\n[raw summary]")
    display(result_obj["summary"].to_frame("value"))

    print("\n[2단계 연결용 핵심 값]")
    print(f"FUEL_KEY = {fuel_key}")
    print(f"BENCHMARK_COL = {frames['benchmark_col']}")
    print(f"CONSUMER_SHIFT_DAYS = {consumer_shift_days}")
    print(f"[저장 폴더] {result_obj['saved_paths']['result_dir']}")

    return output


def run_refinery_weekly_lag(
    fuel_key: str,
    benchmark_col: str | None = None,
    *,
    show_frames: bool = False,
):
    """
    국제제품가격 weekly -> 정유사 세전 공급가격 weekly 시차 분석.

    저장 폴더:
      시차분석/gasoline_refinery_weekly_lag_analysis
      시차분석/diesel_refinery_weekly_lag_analysis
    """
    frames = prepare_stage1_frames(
        fuel_key=fuel_key,
        benchmark_col=benchmark_col,
        verbose=show_frames,
    )

    fuel_label = frames["fuel_label"]

    weekly_upstream_df = frames["weekly_upstream_df"]

    if weekly_upstream_df is None or len(weekly_upstream_df) == 0:
        raise ValueError(f"[{fuel_key}] weekly_upstream_df가 비어 있습니다.")

    required_cols = ["date", "benchmark_weekly", "refinery_pre_tax_weekly"]
    missing_cols = [c for c in required_cols if c not in weekly_upstream_df.columns]
    if missing_cols:
        raise ValueError(f"[{fuel_key}] weekly_upstream_df 필수 컬럼 누락: {missing_cols}")

    exclude_ranges_upstream_weekly = expand_exclude_ranges_for_weekly(
        EXCLUDE_RANGES_STAGE1,
        week_rule="W-SAT",
        buffer_weeks=1,
    )

    upstream_input_df = (
        weekly_upstream_df[["date", "benchmark_weekly", "refinery_pre_tax_weekly"]]
        .copy()
        .rename(columns={
            "benchmark_weekly": "intl_price",
            "refinery_pre_tax_weekly": "dom_price",
        })
        .sort_values("date")
        .reset_index(drop=True)
    )

    upstream_input_df["date"] = pd.to_datetime(upstream_input_df["date"], errors="coerce")
    upstream_input_df["intl_price"] = pd.to_numeric(upstream_input_df["intl_price"], errors="coerce")
    upstream_input_df["dom_price"] = pd.to_numeric(upstream_input_df["dom_price"], errors="coerce")

    upstream_input_df = upstream_input_df.dropna(
        subset=["date", "intl_price", "dom_price"]
    ).reset_index(drop=True)

    upstream_weekly_params = dict(
        start=ANALYSIS_START,
        end=ANALYSIS_END,
        change_method="logdiff",
        exclude_ranges=exclude_ranges_upstream_weekly,
        freq="W-SAT",
        fill_method="ffill",

        max_p=60,
        max_q=60,
        min_train=156,
        response_h=26,

        trend="c",
        causal=False,

        cov_type="HAC",
        hac_lags=4,

        lb_alpha=0.05,
        intl_block_alpha=0.10,
        lb_lag_base=8,

        add_weekday_dummies=False,
        add_month_dummies=True,

        top_n_bic=20,
        show_progress=True,
    )

    print("=" * 100)
    print("[1-B 정유사 주간 시차 입력 데이터]")
    print(f"fuel_key          : {fuel_key}")
    print(f"fuel_label        : {fuel_label}")
    print(f"benchmark_col     : {frames['benchmark_col']}")
    print(f"rows              : {len(upstream_input_df):,}")
    print(f"date range        : {upstream_input_df['date'].min()} ~ {upstream_input_df['date'].max()}")
    print("=" * 100)

    result_obj = run_lag_analysis(
        data=upstream_input_df,
        dom_col="dom_price",
        intl_col="intl_price",
        **upstream_weekly_params,
    )

    shift_periods = int(result_obj["summary"]["recommended_shift_days"])

    result_obj["summary"].loc["fuel_key"] = fuel_key
    result_obj["summary"].loc["fuel_label"] = fuel_label
    result_obj["summary"].loc["benchmark_col"] = frames["benchmark_col"]
    result_obj["summary"].loc["target_layer"] = "refinery_pre_tax_weekly"
    result_obj["summary"].loc["target_description"] = "국제제품가격 주간 -> 정유사 세전 공급가격 주간"
    result_obj["summary"].loc["dom_col"] = "refinery_pre_tax_weekly"
    result_obj["summary"].loc["intl_col"] = "benchmark_weekly"
    result_obj["summary"].loc["freq_unit"] = "weeks"
    result_obj["summary"].loc["recommended_shift_periods"] = shift_periods
    result_obj["summary"].loc["recommended_shift_weeks"] = shift_periods
    result_obj["summary"].loc["recommended_shift_days_equiv"] = shift_periods * 7
    result_obj["summary"].loc["note"] = (
        "recommended_shift_days 컬럼명은 기존 함수 호환 때문에 남아 있으나, "
        "본 정유사 주간 분석에서는 주 단위 lag period로 해석해야 함."
    )

    saved_paths = save_lag_analysis_results(
        result=result_obj,
        root_path=ROOT_DIR,
        result_name=f"{fuel_key}_refinery_weekly_lag_analysis",
        save_change_frame=True,
        save_fixed_frame=True,
        plot_title_prefix=f"{fuel_label} refinery weekly upstream lag",
    )

    result_obj["saved_paths"] = saved_paths

    refinery_shift_weeks = int(result_obj["summary"]["recommended_shift_weeks"])
    refinery_shift_days_equiv = int(result_obj["summary"]["recommended_shift_days_equiv"])

    output = {
        "fuel": fuel_key,
        "fuel_label": fuel_label,
        "benchmark_col": frames["benchmark_col"],
        "consumer_daily_gross_df": frames["consumer_daily_gross_df"],
        "consumer_daily_pretax_df": frames["consumer_daily_pretax_df"],
        "weekly_upstream_df": frames["weekly_upstream_df"],
        "weekly_downstream_df": frames["weekly_downstream_df"],
        "weekly_direct_df": frames["weekly_direct_df"],
        "daily_operating_df": frames["daily_operating_df"],
        "brand_weekly_df": frames["brand_weekly_df"],
        "upstream_result": result_obj,
        "REFINERY_SHIFT_WEEKS": refinery_shift_weeks,
        "REFINERY_SHIFT_DAYS_EQUIV": refinery_shift_days_equiv,
    }

    STAGE1_RESULTS[(fuel_key, "refinery_weekly")] = output

    # 기존 코드 호환: 마지막 실행 결과를 전역 변수에도 남긴다.
    globals()["stage1_outputs"] = output
    globals()["result_obj"] = output

    summary_df = pd.DataFrame([
        stage1_summary_row_safe(
            result_obj,
            f"1-B {fuel_label} refinery pretax weekly",
            "weeks",
        )
    ])

    print("\n[1-B 정유사 주간 시차 분석 summary]")
    display(summary_df)

    print("\n[raw summary]")
    display(result_obj["summary"].to_frame("value"))

    print("\n[2단계 연결용 핵심 값]")
    print(f"FUEL_KEY = {fuel_key}")
    print(f"BENCHMARK_COL = {frames['benchmark_col']}")
    print(f"REFINERY_SHIFT_WEEKS = {refinery_shift_weeks}")
    print(f"REFINERY_SHIFT_DAYS_EQUIV = {refinery_shift_days_equiv}")
    print(f"[저장 폴더] {saved_paths['result_dir']}")

    return output


print("[READY] 1단계 분석 실행 함수 정의 완료")

## 분석 A. 주유소 휘발유 시차


In [ ]:
# =========================================================
# 분석 A) 주유소 휘발유 시차
# 국제제품가격 daily -> 주유소 소비자가격 daily
# =========================================================

FUEL_KEY = "gasoline"
BENCHMARK_COL_LOCAL = "휘발유92RON_원리터"
# BENCHMARK_COL_LOCAL = None  # 기본값 사용 시

gasoline_consumer_stage1 = run_consumer_daily_lag(
    fuel_key=FUEL_KEY,
    benchmark_col=BENCHMARK_COL_LOCAL,
    show_frames=False,
)

## 분석 B. 주유소 경유 시차


In [ ]:
# =========================================================
# 분석 B) 주유소 경유 시차
# 국제제품가격 daily -> 주유소 소비자가격 daily
# =========================================================

FUEL_KEY = "diesel"
BENCHMARK_COL_LOCAL = "경유0.001_원리터"
# BENCHMARK_COL_LOCAL = "경유0.05_원리터"
# BENCHMARK_COL_LOCAL = None  # 기본값 사용 시

diesel_consumer_stage1 = run_consumer_daily_lag(
    fuel_key=FUEL_KEY,
    benchmark_col=BENCHMARK_COL_LOCAL,
    show_frames=False,
)

## 분석 C. 정유사 휘발유 주간 시차


In [ ]:
# =========================================================
# 분석 C) 정유소 휘발유 시차
# 국제제품가격 weekly -> 정유사 세전 공급가격 weekly
# =========================================================

FUEL_KEY = "gasoline"
BENCHMARK_COL_LOCAL = "휘발유92RON_원리터"
# BENCHMARK_COL_LOCAL = None  # 기본값 사용 시

gasoline_refinery_stage1 = run_refinery_weekly_lag(
    fuel_key=FUEL_KEY,
    benchmark_col=BENCHMARK_COL_LOCAL,
    show_frames=False,
)

## 분석 D. 정유사 경유 주간 시차


In [ ]:
# =========================================================
# 분석 D) 정유소 경유 시차
# 국제제품가격 weekly -> 정유사 세전 공급가격 weekly
# =========================================================

FUEL_KEY = "diesel"
BENCHMARK_COL_LOCAL = "경유0.001_원리터"
# BENCHMARK_COL_LOCAL = "경유0.05_원리터"
# BENCHMARK_COL_LOCAL = None  # 기본값 사용 시

diesel_refinery_stage1 = run_refinery_weekly_lag(
    fuel_key=FUEL_KEY,
    benchmark_col=BENCHMARK_COL_LOCAL,
    show_frames=False,
)